# Database Face Embeddings

## Project Objective

The goal of this stage is to convert reference photographs of missing
persons into **ArcFace face embeddings** and store them for efficient
future face searches.

The original photographs will be preserved as reference evidence,
while embeddings will provide the searchable representation.

## Pipeline

```text
Reference Photograph
        │
        ▼
  Face Detection
        │
        ▼
    ArcFace Model
        │
        ▼
  512-D Embedding
        │
        ▼
 Store Embedding
        │
        ▼
Reusable Face Database

## Database Structure

Each missing person can have multiple reference photographs.

```text
Missing Person
│
├── Person ID
├── Personal Information
├── Report Information
│
├── Reference Images
│   ├── image_01.jpg
│   ├── image_02.jpg
│   └── image_03.jpg
│
└── Face Embeddings
    ├── embedding_01
    ├── embedding_02
    └── embedding_03

In [1]:
import os
import numpy as np
import pandas as pd

from deepface import DeepFace

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
DATABASE_DIR = "face_database_test"

print("Database directory:", DATABASE_DIR)
print("Database exists:", os.path.exists(DATABASE_DIR))

Database directory: face_database_test
Database exists: True


In [3]:
people = sorted([
    person
    for person in os.listdir(DATABASE_DIR)
    if os.path.isdir(os.path.join(DATABASE_DIR, person))
])

print("Number of people:", len(people))
print("People:", people)

total_images = 0

for person in people:
    person_path = os.path.join(DATABASE_DIR, person)

    images = [
        image
        for image in os.listdir(person_path)
        if image.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    total_images += len(images)

    print(f"{person}: {len(images)} image(s)")

print("Total reference images:", total_images)

Number of people: 8
People: ['person1', 'person2', 'person3', 'person4', 'person5', 'person6', 'person7', 'person8']
person1: 7 image(s)
person2: 2 image(s)
person3: 3 image(s)
person4: 2 image(s)
person5: 2 image(s)
person6: 2 image(s)
person7: 2 image(s)
person8: 2 image(s)
Total reference images: 22


In [4]:
embedding_records = []

for person in people:
    person_path = os.path.join(DATABASE_DIR, person)

    images = sorted([
        image
        for image in os.listdir(person_path)
        if image.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    for image_name in images:
        image_path = os.path.join(person_path, image_name)

        embedding_result = DeepFace.represent(
            img_path=image_path,
            model_name="ArcFace",
            detector_backend="opencv",
            enforce_detection=True
        )

        embedding = np.array(
            embedding_result[0]["embedding"]
        )

        embedding_records.append({
            "person": person,
            "image": image_name,
            "embedding": embedding
        })

print("Embeddings generated:", len(embedding_records))
print("Embedding size:", embedding_records[0]["embedding"].shape)

Embeddings generated: 22
Embedding size: (512,)


In [5]:
embedding_df = pd.DataFrame(embedding_records)

print("Number of records:", len(embedding_df))
print("Columns:", embedding_df.columns.tolist())

display(embedding_df[["person", "image"]].head())

Number of records: 22
Columns: ['person', 'image', 'embedding']


,person,image
0,person1,img10.jpg
1,person1,img11.jpg
2,person1,img2.jpg
3,person1,img4.jpg
4,person1,img5.jpg


In [10]:
embedding_data = np.array(
    [record["embedding"] for record in embedding_records]
)

metadata_array = metadata.to_numpy(dtype="<U50")

np.savez(
    "face_embeddings.npz",
    embeddings=embedding_data,
    metadata=metadata_array
)

print("Embeddings database saved again!")
print("Embedding matrix:", embedding_data.shape)
print("Metadata matrix:", metadata_array.shape)

Embeddings database saved again!
Embedding matrix: (22, 512)
Metadata matrix: (22, 2)


In [11]:
saved_data = np.load("face_embeddings.npz")

loaded_embeddings = saved_data["embeddings"]
loaded_metadata = saved_data["metadata"]

print("Loaded embeddings shape:", loaded_embeddings.shape)
print("Loaded metadata shape:", loaded_metadata.shape)

print("\nFirst record:")
print("Person:", loaded_metadata[0][0])
print("Image:", loaded_metadata[0][1])

Loaded embeddings shape: (22, 512)
Loaded metadata shape: (22, 2)

First record:
Person: person1
Image: img10.jpg


In [14]:
# Generate embedding only for the new query
query_embedding_result = DeepFace.represent(
    img_path=held_out_query_path,
    model_name="ArcFace",
    detector_backend="opencv",
    enforce_detection=True
)

query_embedding = np.array(
    query_embedding_result[0]["embedding"]
)

print("Query embedding shape:", query_embedding.shape)
print("Stored database embeddings:", loaded_embeddings.shape)

Query embedding shape: (512,)
Stored database embeddings: (22, 512)


In [13]:
held_out_query_path = "/tmp/img1_query.jpg"

print("Held-out query:", held_out_query_path)
print("Image exists:", os.path.exists(held_out_query_path))

Held-out query: /tmp/img1_query.jpg
Image exists: True


In [16]:
def cosine_distance(embedding_a, embedding_b):
    similarity = np.dot(embedding_a, embedding_b) / (
        np.linalg.norm(embedding_a) * np.linalg.norm(embedding_b)
    )

    distance = 1 - similarity
    return distance


print("Cosine distance function ready!")

Cosine distance function ready!


In [17]:
search_results = []

for index, database_embedding in enumerate(loaded_embeddings):

    distance = cosine_distance(
        query_embedding,
        database_embedding
    )

    person = loaded_metadata[index][0]
    image = loaded_metadata[index][1]

    search_results.append({
        "person": person,
        "image": image,
        "distance": distance
    })

search_results = sorted(
    search_results,
    key=lambda x: x["distance"]
)

print("Top 5 matches:")
print()

for result in search_results[:5]:
    print(
        f"{result['person']} | "
        f"{result['image']} | "
        f"distance: {result['distance']:.4f}"
    )

Top 5 matches:

person1 | img4.jpg | distance: 0.4969
person1 | img2.jpg | distance: 0.5078
person1 | img11.jpg | distance: 0.5182
person1 | img7.jpg | distance: 0.5230
person1 | img5.jpg | distance: 0.5399


In [18]:
stored_person_results = {}

for result in search_results:
    person = result["person"]
    distance = result["distance"]

    if person not in stored_person_results:
        stored_person_results[person] = []

    stored_person_results[person].append(distance)

stored_ranked_people = []

for person, distances in stored_person_results.items():
    stored_ranked_people.append({
        "person": person,
        "best_distance": min(distances),
        "average_distance": np.mean(distances),
        "photo_count": len(distances)
    })

stored_ranked_people = sorted(
    stored_ranked_people,
    key=lambda x: x["best_distance"]
)

print("Person-level ranking using stored embeddings:")
print()

for rank, candidate in enumerate(stored_ranked_people, start=1):
    print(
        f"{rank}. {candidate['person']} | "
        f"best: {candidate['best_distance']:.4f} | "
        f"average: {candidate['average_distance']:.4f} | "
        f"photos: {candidate['photo_count']}"
    )

Person-level ranking using stored embeddings:

1. person1 | best: 0.4969 | average: 0.5463 | photos: 7
2. person8 | best: 0.8300 | average: 0.8344 | photos: 2
3. person7 | best: 0.8723 | average: 0.9018 | photos: 2
4. person2 | best: 0.9216 | average: 0.9686 | photos: 2
5. person4 | best: 0.9506 | average: 0.9700 | photos: 2
6. person5 | best: 0.9633 | average: 0.9884 | photos: 2
7. person6 | best: 0.9712 | average: 1.0106 | photos: 2
8. person3 | best: 1.0460 | average: 1.1096 | photos: 3


# Conclusion

A precomputed face-embedding database was successfully implemented
using ArcFace.

The system processes reference photographs once and generates a
512-dimensional embedding for each detected face. These embeddings are
stored in `face_embeddings.npz` along with the corresponding person
and image metadata.

### Results

- **People in database:** 8
- **Reference images:** 22
- **Embedding dimension:** 512
- **Stored embedding matrix:** 22 × 512

### Search Test

A held-out image of `person1` was used as a new query. The query image
was not present in the stored database.

The system generated only one new query embedding and compared it
against the 22 precomputed embeddings.

- **Expected person:** person1
- **Predicted person:** person1
- **Best distance:** 0.4969
- **Average distance:** 0.5463

The stored-embedding search produced the same correct ranking as the
previous image-by-image search.

### Engineering Improvement

The previous approach generated embeddings for every database image
during every search. The new approach generates database embeddings
only once and reuses them.

This makes the system more efficient and provides a foundation for
scaling the face-search system to a larger missing-person database.

### Limitation

The current `.npz` storage is a prototype solution. A real-world
application will require persistent database storage, image
management, person metadata, access control, and scalable similarity
search.

The search threshold also requires proper evaluation and calibration
before being used in a real-world deployment.

### Next Stage

The next stage will focus on evaluating the face-search system more
systematically and improving the person-level matching strategy before
building the application layer.